# Exploração dos primos gerados por $P(b, n) = (b^n) \bmod ((b-1)^n)$

O objetivo deste notebook é localizar combinações de inteiros `b` (base) e `n` (expoente)
para as quais o valor
\[ P(b, n) = (b^n) \bmod ((b-1)^n) \]
seja um número primo. Basta executar as células em ordem (sem necessidade de comandos
`git apply` ou similares) para realizar a busca diretamente no Google Colab.


In [ ]:
from typing import List, Tuple

def miller_rabin(n: int) -> bool:
    """Teste determinístico de primalidade para inteiros de 64 bits.

    Para números maiores o teste segue altamente confiável por utilizar uma combinação
    de bases conhecidas.
    """
    if n < 2:
        return False
    small_primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    if n in small_primes:
        return True
    if any(n % p == 0 for p in small_primes):
        return False

    # escreve n - 1 como d * 2^s
    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1

    def try_composite(a: int) -> bool:
        x = pow(a, d, n)
        if x in (1, n - 1):
            return False
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                return False
        return True

    # Bases suficientes para todos os inteiros de 64 bits
    test_bases = (2, 3, 5, 7, 11, 13, 17)

    return not any(try_composite(a) for a in test_bases if a < n)

def calcular_P(b: int, n: int) -> int:
    """Calcula P(b, n) = (b**n) mod ((b-1)**n)."""
    if b <= 1:
        raise ValueError("A base b deve ser maior que 1.")
    if n < 1:
        raise ValueError("O expoente n deve ser positivo.")
    modulo = pow(b - 1, n)
    return pow(b, n, modulo)

def gerar_primos(limit_b: int, limit_n: int) -> List[Tuple[int, int, int]]:
    """Gera todos os pares (b, n) até os limites dados em que P(b, n) é primo."""
    resultados: List[Tuple[int, int, int]] = []
    for b in range(2, limit_b + 1):
        for n in range(1, limit_n + 1):
            valor = calcular_P(b, n)
            if valor > 1 and miller_rabin(valor):
                resultados.append((b, n, valor))
    return resultados

def primeiros_primos(limit_b: int, limit_n: int, quantidade: int) -> List[Tuple[int, int, int]]:
    """Retorna as primeiras ocorrências onde P(b, n) é primo.

    A busca é feita crescendo b e, para cada b, crescendo n.
    """
    encontrados: List[Tuple[int, int, int]] = []
    for b in range(2, limit_b + 1):
        for n in range(1, limit_n + 1):
            valor = calcular_P(b, n)
            if valor > 1 and miller_rabin(valor):
                encontrados.append((b, n, valor))
                if len(encontrados) >= quantidade:
                    return encontrados
    return encontrados


In [ ]:
# Ajuste os valores de b e n conforme necessário
b = 4
n = 2
valor = calcular_P(b, n)
print(f"P({b}, {n}) = {valor}")
print("É primo?", miller_rabin(valor))


In [ ]:
# Exemplo de busca: ajuste os limites e a quantidade desejada
limite_b = 15
limite_n = 10
quantidade = 10
primos_encontrados = primeiros_primos(limite_b, limite_n, quantidade)
for b, n, valor in primos_encontrados:
    print(f"b={b}, n={n} => P(b, n) = {valor}")

if not primos_encontrados:
    print("Nenhum P(b, n) primo encontrado nos intervalos informados.")
